# RiskBrief (GeoRisk Copilot): prototype notebook

**Team GeoRisk Copilot**: Yashej Yang Shah (S24CSEU0256), Tanu Somani (S24CSEU0355). SCSE4041 Mini Project, Milestone 2.

RiskBrief reads the JSON returned by our GeoRisk fraud/RTO scoring API for one order and produces:
1. a **decision brief** for the shop's order reviewer (action from the store playbook, reasons in plain words, every number taken from the input),
2. a polite **customer message draft** in English, and in Hindi when the reviewer selects it,
3. a **check report** from code checks C1-C11. Failed outputs get one repair try (P4), then a safe fixed template.

A person always approves before anything is sent (sending is simulated).

**How to use this notebook**
- The outputs below are saved from a real run (Qwen 2.5 7B on Ollama, local RTX 4070 laptop GPU).
- To run it yourself in Colab: *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*. The first cell installs Ollama and downloads the model (about 5 GB, a few minutes).
- Change the JSON in the last section to try your own order.

Code, prompts and test data: https://github.com/YashejShah/riskbrief-georisk-copilot

In [1]:
import os, subprocess, sys, time, json, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("riskbrief-georisk-copilot"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/YashejShah/riskbrief-georisk-copilot.git"], check=True)
    os.chdir("riskbrief-georisk-copilot")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    subprocess.run(["ollama", "pull", "qwen2.5:7b"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jsonschema"], check=True)
elif os.path.basename(os.getcwd()) == "code":       # opened from the code/ folder of the repository
    os.chdir("..")
sys.path.insert(0, os.path.join(os.getcwd(), "code"))
from riskbrief import chain
from riskbrief.llm import MODEL, SETTINGS, model_digest
print("running in Colab" if IN_COLAB else "running locally", "| model:", MODEL, model_digest(), "| settings:", SETTINGS)

running locally | model: qwen2.5:7b 845dbda0ea48 | settings: {'temperature': 0, 'seed': 42, 'num_ctx': 8192}


## 1. The test inputs
15 inputs: 7 normal orders, 5 edge cases and 3 invalid inputs. Normal orders are real outputs of the GeoRisk API for
synthetic orders. Edge and invalid cases start from a real API output and change one thing on purpose (see `note`).

In [2]:
cases = json.load(open("test-data/cases.json", encoding="utf-8"))
for c in cases:
    r = c["response"]
    print(f'{c["id"]}  {c["type"]:<7} decision={str(r.get("decision")):<8} score={str(r.get("risk_score")):>4}  '
          f'codes={r.get("reason_codes")}  {c["title"]}')

T01  Normal  decision=approve  score=   8  codes=[]  Small UPI order, same city, approve, no codes (R1)
T02  Normal  decision=review   score=  67  codes=['ODD_HOUR', 'LARGE_BILLING_SHIPPING_DISTANCE', 'HIGH_AMOUNT']  High-value COD at 3 AM, far shipping, review (R3)
T03  Normal  decision=approve  score=   4  codes=['LARGE_BILLING_SHIPPING_DISTANCE']  Card order, Mumbai billing to Kolkata shipping, approve with a code (R2)
T04  Normal  decision=review   score=  56  codes=['ODD_HOUR']  Prepaid (UPI) order under review (R4)
T05  Normal  decision=review   score=  56  codes=['ODD_HOUR', 'LARGE_BILLING_SHIPPING_DISTANCE', 'HIGH_AMOUNT']  Wallet order under review (R4, wallet is a risky method)
T06  Normal  decision=review   score=  68  codes=['HIGH_AMOUNT', 'RISKY_PAYMENT_METHOD']  COD order under review, daytime (R3)
T07  Edge    decision=decline  score=  91  codes=['ODD_HOUR', 'LARGE_BILLING_SHIPPING_DISTANCE', 'HIGH_AMOUNT']  Order the API declines (R6: a person decides, the model never c

## 2. Run the final prompt chain on four inputs
T02 (COD order under review), T11 (region-only review, fairness rule R5), T12 (same order, Hindi selected) and
T10 (prompt injection hidden in the order ID).

In [3]:
def show(res):
    fin, b = res["final"], res["final"]["brief"]
    print(f"ACTION: {b.get('recommended_action')} (rule {b.get('playbook_rule')}) | API decision {b.get('api_decision')} "
          f"| score {b.get('risk_score')} | raw LLM action {res['raw_action']}")
    print("SUMMARY:", b.get("summary"))
    for r in b.get("reasons", []):
        print(f"  - {r['code']}: {r['plain_language']}")
    if fin.get("message_en"):
        print("EN DRAFT:", fin["message_en"]["message"])
    if fin.get("message_hi"):
        print("HI DRAFT:", fin["message_hi"]["message"])
    fails = [f"{s} {k}: {e}" for s, d in res["errors"].items() for k, v in d.items() for e in v]
    print("CHECKS:", "all passed" if not fails else fails)
    for e in res.get("events", []):
        print("EVENT:", e)
    print("LLM calls:", len(res["calls"]), "| seconds:", round(sum(c["stats"]["seconds"] for c in res["calls"]), 1))

by_id = {c["id"]: c for c in cases}
for cid in ["T02", "T11", "T12", "T10"]:
    print("=" * 30, cid, by_id[cid]["title"])
    show(chain.run(by_id[cid], "final"))

============================== T02 High-value COD at 3 AM, far shipping, review (R3)


ACTION: OTP_VERIFY (rule R3) | API decision review | score 67 | raw LLM action OTP_VERIFY
SUMMARY: Risk score 67 out of 100, API decision: review. Main reasons: the order was placed at 03:10, before 06:00, the shipping address is more than 300 km from the billing address, and the order amount is Rs 89,999, which is Rs 5,000 or more. Ask the customer to confirm the order with an OTP before you ship. If the OTP is not confirmed, call the customer. You may also offer online payment, but cash on delivery stays available.
  - ODD_HOUR: The order was placed at 03:10, before 06:00.
  - LARGE_BILLING_SHIPPING_DISTANCE: The shipping address is more than 300 km from the billing address.
  - HIGH_AMOUNT: The order amount is Rs 89,999, which is Rs 5,000 or more.
EN DRAFT: Thank you for your high-value order PE-002! We will send you a one-time password (OTP) to confirm it. You may also pay online by UPI or card if you prefer, and cash on delivery is still available. - {store_name}
CHECKS: all passe

ACTION: OTP_VERIFY (rule R5) | API decision review | score 83 | raw LLM action OTP_VERIFY
SUMMARY: Risk score 83 out of 100, API decision: review. The API's fairness rule changed a borderline decline into a review. One reason is a statistic about the billing state; see the note below. Ask the customer to confirm the order with an OTP before you ship.
  - HIGH_RISK_REGION: The billing state's rate in the model's synthetic training data is above average. This alone is never a reason to decline, block or cancel an order.
EN DRAFT: Thank you for your order PE-002! We will send you a one-time password (OTP) to confirm it. - {store_name}
CHECKS: all passed
LLM calls: 2 | seconds: 6.0
============================== T12 Same review order, Hindi message selected by the reviewer (P3)


ACTION: OTP_VERIFY (rule R3) | API decision review | score 67 | raw LLM action OTP_VERIFY
SUMMARY: Risk score 67 out of 100, API decision: review. Main reasons: the order was placed at 03:10, before 06:00, the shipping address is more than 300 km from the billing address, and the order amount is Rs 89,999, which is Rs 5,000 or more. Ask the customer to confirm the order with an OTP before you ship. If the OTP is not confirmed, call the customer. You may also offer online payment, but cash on delivery stays available.
  - ODD_HOUR: The order was placed at 03:10, before 06:00.
  - LARGE_BILLING_SHIPPING_DISTANCE: The shipping address is more than 300 km from the billing address.
  - HIGH_AMOUNT: The order amount is Rs 89,999, which is Rs 5,000 or more.
EN DRAFT: Thank you for your high-value order PE-002! We will send you a one-time password (OTP) to confirm it. You may also pay online by UPI or card if you prefer, and cash on delivery is still available. - {store_name}
HI DRAFT: आपके ज़

ACTION: OTP_VERIFY (rule R3) | API decision review | score 67 | raw LLM action OTP_VERIFY
SUMMARY: Risk score 67 out of 100, API decision: review. Main reasons: the order was placed at 03:10, before 06:00, the shipping address is more than 300 km from the billing address, and the order amount is Rs 89,999, which is Rs 5,000 or more. Ask the customer to confirm the order with an OTP before you ship. If the OTP is not confirmed, call the customer. You may also offer online payment, but cash on delivery stays available.
  - ODD_HOUR: The order was placed at 03:10, before 06:00.
  - LARGE_BILLING_SHIPPING_DISTANCE: The shipping address is more than 300 km from the billing address.
  - HIGH_AMOUNT: The order amount is Rs 89,999, which is Rs 5,000 or more.
EN DRAFT: Thank you for your high-value order PE-002! We will send you a one-time password (OTP) to confirm it. You may also pay online by UPI or card if you prefer, and cash on delivery is still available. - {store_name}
CHECKS: all passe

## 3. Invalid input: the chain stops before the LLM
T16 has a broken API response (decision missing, score not a number). RiskBrief does not call the model and asks a
person to look at it (playbook rule R8).

In [4]:
show(chain.run(by_id["T16"], "final"))

ACTION: NEEDS_HUMAN (rule R8) | API decision None | score None | raw LLM action NEEDS_HUMAN
SUMMARY: This order needs a person to look at it. RiskBrief did not call the LLM because: decision missing or unknown (None); risk_score must be an integer 0-100 (got 'high')
CHECKS: all passed
LLM calls: 0 | seconds: 0


## 4. Results of every prompt version on all 15 inputs
Saved by `code/run_eval.py` (same model, temperature 0, seed 42). v1 is the Milestone 1 baseline prompt. T01-T17
were used while improving the prompts; `final_heldout` is 6 new inputs (H01-H06) written after the final prompt was
frozen and run once.

In [5]:
for v in ["v1", "v2", "v3", "v4", "v5", "v6", "final", "final_heldout"]:
    p = f"test-data/outputs/{v}/summary.json"
    if os.path.exists(p):
        s = json.load(open(p, encoding="utf-8"))
        print(f"{v:<6} all checks pass {s['all_checks_pass']:>2}/{s['cases']}   raw LLM action = playbook "
              f"{s['raw_action_ok']:>2}/{s['cases']}   {s['mean_seconds_per_case']} s per order")

v1     all checks pass  0/17   raw LLM action = playbook 11/17   2.0 s per order
v2     all checks pass  6/17   raw LLM action = playbook 17/17   6.5 s per order
v3     all checks pass 10/17   raw LLM action = playbook 17/17   8.1 s per order
v4     all checks pass 15/17   raw LLM action = playbook 17/17   8.8 s per order
v5     all checks pass 15/17   raw LLM action = playbook 17/17   8.1 s per order
v6     all checks pass 14/17   raw LLM action = playbook 17/17   9.3 s per order
final  all checks pass 17/17   raw LLM action = playbook 17/17   8.1 s per order
final_heldout all checks pass  6/6   raw LLM action = playbook  6/6   8.8 s per order


## 5. Try your own order
Edit the JSON below (keep the same fields) and run the cell. Use made-up orders only.

In [6]:
my_case = {
  "request": {"order_id": "MY-001", "amount": 6500, "payment_method": "cod", "timestamp": "2026-11-15T04:20:00+05:30",
              "customer": {"billing_pincode": "560001", "shipping_pincode": "560034"}},
  "response": {"order_id": "MY-001", "risk_score": 62, "decision": "review",
               "reason_codes": ["ODD_HOUR", "HIGH_AMOUNT"],
               "geo_signals": {"billing_shipping_distance_km": 0.0, "region_risk": 0.01133, "income_tier": "tier1",
                               "billing_district": "Bengaluru", "billing_state": "Karnataka"},
               "model_version": "india-synth-0.1",
               "fairness_note": "No fairness adjustment needed; decision is not a decline."},
  "language": "hi"}
show(chain.run(my_case, "final"))

ACTION: OTP_VERIFY (rule R3) | API decision review | score 62 | raw LLM action OTP_VERIFY
SUMMARY: Risk score 62 out of 100, API decision: review. Main reasons: the order was placed at 04:20, before 06:00, the order amount is Rs 6,500, which is Rs 5,000 or more. Ask the customer to confirm the order with an OTP before you ship. If the OTP is not confirmed, call the customer. You may also offer online payment, but cash on delivery stays available.
  - ODD_HOUR: The order was placed at 04:20, before 06:00.
  - HIGH_AMOUNT: The order amount is Rs 6,500, which is Rs 5,000 or more.
EN DRAFT: Thank you for your high-value order MY-001! We will send you a one-time password (OTP) to confirm it. You may also pay online by UPI or card if you prefer, and cash on delivery is still available. - {store_name}
HI DRAFT: आपके ज़्यादा कीमत वाले ऑर्डर MY-001 के लिए धन्यवाद! इसे पक्का करने के लिए हम आपको एक वन-टाइम पासवर्ड (OTP) भेजेंगे। आप चाहें तो UPI या कार्ड से ऑनलाइन भुगतान भी कर सकते हैं, और कैश ऑन 